In [0]:
import pyspark.sql.functions as F

catalog_name = "f1_dev"
target_table = f"{catalog_name}.silver.dim_races"



In [0]:
# 1. Read raw races table
df_races_raw = spark.table(f"{catalog_name}.bronze.races")



In [0]:
# 2. Transform & Cleanse
df_races_silver = (
    df_races_raw
    .replace(r"\N", None)
    .select(
        F.col("raceId").cast("integer").alias("race_id"),
        F.col("year").cast("integer").alias("race_year"),
        F.col("round").cast("integer").alias("race_round"),
        F.col("circuitId").cast("integer").alias("circuit_id"),
        F.col("name").alias("race_name"),
        F.to_date(F.col("date"), "yyyy-MM-dd").alias("race_date")
    )
)



In [0]:
# 3. Write as Delta table in Silver schema
df_races_silver.write.mode("overwrite").saveAsTable(target_table)



In [0]:
# 4. Set Unity Catalog Governance Tags
spark.sql(f"""
    ALTER TABLE {target_table} SET TAGS (
        'layer' = 'silver',
        'table_type' = 'dimension',
        'contains_pii' = 'false'
    )
""")

print(f"✓ Successfully built and governed: {target_table}")